In [1]:
import torch
import torch.nn as nn
import torchvision.transforms.functional as func

from torchvision.models import resnet18

class ResNet18Encoder(nn.Module):
    def __init__(self):
        super().__init__()
        resnet = resnet18(weights=None)
        self.stem = nn.Sequential(
            nn.Conv2d(6, 64, kernel_size=7, stride=2, padding=3, bias=False),
            resnet.bn1,
            resnet.relu,
        )
        self.pool = resnet.maxpool
        self.encoder1 = resnet.layer1
        self.encoder2 = resnet.layer2
        self.encoder3 = resnet.layer3
        self.encoder4 = resnet.layer4
    def forward(self, x):
        x0 = self.stem(x)
        x1 = self.encoder1(self.pool(x0))
        x2 = self.encoder2(x1)
        x3 = self.encoder3(x2)
        x4 = self.encoder4(x3)
        return x0, x1, x2, x3, x4


class DecoderBlock(nn.Module):
    def __init__(self, in_channels, skip_channels, out_channels):
        super().__init__()
        self.up = nn.ConvTranspose2d(in_channels, out_channels, kernel_size=2, stride=2)
        self.conv = nn.Sequential(
            nn.Conv2d(skip_channels + out_channels, out_channels, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
        )

    def forward(self, x, skip):
        x = self.up(x)
        skip = func.center_crop(skip, [x.shape[2], x.shape[3]])
        x = torch.cat([x, skip], dim=1)
        return self.conv(x)

class ResNetUNet(nn.Module):
    def __init__(self, n_classes=1):
        super().__init__()
        self.encoder = ResNet18Encoder()

        self.decoder4 = DecoderBlock(512, 256, 256)
        self.decoder3 = DecoderBlock(256, 128, 128)
        self.decoder2 = DecoderBlock(128, 64, 64)
        self.decoder1 = DecoderBlock(64, 64, 32)

        self.final_conv = nn.Conv2d(32, n_classes, kernel_size=1)

    def forward(self, x):
        x0, x1, x2, x3, x4 = self.encoder(x)

        d4 = self.decoder4(x4, x3)
        d3 = self.decoder3(d4, x2)
        d2 = self.decoder2(d3, x1)
        d1 = self.decoder1(d2, x0)

        out = self.final_conv(d1)
        return out

In [ ]:
from pytorch_wavelets import DWTForward, DWTInverse

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

dwt = DWTForward(J=1, wave='haar', mode='reflect').to(device)
idwt = DWTInverse(wave='haar', mode='reflect').to(device)

@torch.no_grad()
def wavelet_decompose(img):       
    img = img.to(device)
    img = img.unsqueeze(0)       
    LL, H = dwt(img)               

    H0 = H[0]                   

    LH = H0[:,:,0,:,:].squeeze(0)  
    HL = H0[:,:,1,:,:].squeeze(0)  
    HH = H0[:,:,2,:,:].squeeze(0)  

    return LL.squeeze(0), LH, HL, HH

@torch.no_grad()
def make_LL_only(img):
    img = img.to(device)
    LL, LH, HL, HH = wavelet_decompose(img)  


    zeros = torch.zeros_like(LH)        
    H0 = torch.stack([zeros, zeros, zeros], dim=1)  
    H_list = [H0.unsqueeze(0)]            

    out = idwt((LL.unsqueeze(0), H_list)) 
    return out.squeeze(0).clamp(0, 1)     


@torch.no_grad()
def make_HF_only(img):
    LL, LH, HL, HH = wavelet_decompose(img)

    zeros_LL = torch.zeros_like(LL)          
    H0 = torch.stack([LH, HL, HH], dim=1)    
    H_list = [H0.unsqueeze(0)]               

    out = idwt((zeros_LL.unsqueeze(0), H_list))
    return out.squeeze(0).clamp(0, 1)

@torch.no_grad()
def rgb_plus_LL(img):
    img = img.to(device)
    LL, LH, HL, HH = wavelet_decompose(img)
    LL_up = torch.nn.functional.interpolate(LL.unsqueeze(0), size=img.shape[1:], mode='bilinear', align_corners=False)
    out = torch.cat([img.unsqueeze(0), LL_up], dim=1).squeeze(0) 
    return out.cpu()

In [ ]:
from torch.utils.data import Dataset
import tifffile as tif
import random
from torchvision import transforms
import os

random.seed(42)

imageTransform = transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))])
maskTransform = transforms.Compose([transforms.ToTensor()])

class LandslideDataset(Dataset):
    def __init__(self, img_list, mask_list):
        self.img_list = img_list
        self.mask_list = mask_list
    
    def __len__(self):
        return len(self.img_list)
    
    def __getitem__(self, index):
        img_dir = self.img_list[index]
        mask_dir = self.mask_list[index]

        img = tif.imread(img_dir)
        mask = tif.imread(mask_dir)

        img = imageTransform(img)
        mask = (maskTransform(mask) > 0).float()

        return img, mask
    
class LLOnlyDataset(Dataset):
    def __init__(self, img_list, mask_list):
        self.img_list = img_list
        self.mask_list = mask_list
        self.normalize = transforms.Normalize((0.5,0.5,0.5),(0.5,0.5,0.5))
        
    def __len__(self):
        return len(self.img_list)

    def __getitem__(self, idx):
        img = tif.imread(self.img_list[idx])
        if isinstance(img, tuple):
            img = img[0]
        img = torch.from_numpy(img / 255.0).float().permute(2,0,1)
        LLimg = make_LL_only(img).cpu()                      # <— NEW
        LLimg = self.normalize(LLimg)
        mask = tif.imread(self.mask_list[idx])
        if isinstance(mask, tuple):
            mask = mask[0]
        mask = (maskTransform(mask) > 0).float()        
        return LLimg, mask
    
class HFOnlyDataset(Dataset):
    def __init__(self, img_list, mask_list):
        self.img_list = img_list
        self.mask_list = mask_list
        self.normalize = transforms.Normalize((0.5,0.5,0.5),(0.5,0.5,0.5))
        
    def __len__(self):
        return len(self.img_list)

    def __getitem__(self, idx):
        img = tif.imread(self.img_list[idx])
        if isinstance(img, tuple):
            img = img[0]
        img = torch.from_numpy(img / 255.0).float().permute(2,0,1)
        HFimg = make_HF_only(img).cpu()                      # <— NEW
        HFimg = self.normalize(HFimg)
        mask = tif.imread(self.mask_list[idx])
        if isinstance(mask, tuple):
            mask = mask[0]
        mask = (maskTransform(mask) > 0).float()        
        return HFimg, mask
    
class RGBLLDataset(Dataset):
    def __init__(self, img_list, mask_list):
        self.img_list = img_list
        self.mask_list = mask_list
        
    def __len__(self):
        return len(self.img_list)

    def __getitem__(self, idx):
        img = tif.imread(self.img_list[idx])
        img = torch.from_numpy(img / 255.0).float().permute(2,0,1)
        
        img_aug = rgb_plus_LL(img).cpu()                
        
        mean = torch.tensor([0.5]*6).view(6,1,1)
        std = torch.tensor([0.5]*6).view(6,1,1)
        img_aug = (img_aug.cpu() - mean) / std
        
        mask = tif.imread(self.mask_list[idx])
        mask = (maskTransform(mask) > 0).float()
        return img_aug, mask

path = "../../data/"
limit = 100
regions = [f for f in os.listdir(path) if (os.path.isdir(os.path.join(path, f)) and f != "study areas shp")]

regions_dict = dict()
regions_pos_weight = dict()

for region in regions:
    dataset_dir = path + region
    image_dir = os.path.join(dataset_dir, "img")
    img_list = os.listdir(image_dir)

    all_images = sorted(os.path.join(image_dir, f) for f in img_list)

    if len(all_images) > limit:
        all_images = random.sample(all_images, limit)

    regions_dict[region] = all_images
    regions_pos_weight[region] = 0

def estimate_pos_weight(mask_paths):
    pos, tot = 0, 0
    for path in mask_paths:
        m = tif.imread(path)
        m = torch.from_numpy((m > 0).astype("float32"))
        pos += m.sum().item()
        tot += m.numel()
    neg = max(tot - pos, 1.0)
    pos = max(pos, 1.0)
    return neg / pos

# for region in regions_dict:
#     masks = [f.replace("img", "mask") for f in regions_dict[region]]
#
#     regions_pos_weight[region] = estimate_pos_weight(masks)

In [4]:
import numpy as np
import torch.nn.functional as F

### TUNE THIS CONSTANT
threshold = 0.6

def getMetrics(TP, TN, FP, FN):
    precision = TP / (TP + FP + 1e-8)
    recall = TP / (TP + FN + 1e-8)
    f1 = 2 * precision * recall / (precision + recall + 1e-8)
        
    iou_1  = TP / (TP + FP + FN + 1e-8)
    iou_0 = TN / (TN + FP + FN + 1e-8)
    miou = (iou_0 + iou_1) / 2 
        
    oa = (TP + TN)/(TP + TN + FP + FN + 1e-8)
    
    return precision, recall, f1, iou_1, miou, oa

def dice_loss(pred, target, smooth=1.):
    pred = torch.sigmoid(pred)
    pred_flat = pred.view(-1)
    target_flat = target.view(-1)
    intersection = (pred_flat * target_flat).sum()
    return 1 - ((2. * intersection + smooth) / (pred_flat.sum() + target_flat.sum() + smooth))

def boundary_f1_score(pred_mask, true_mask, tolerance=3):
    if isinstance(pred_mask, torch.Tensor):
        pred = pred_mask.detach().cpu().float()
    else:
        pred = torch.from_numpy(pred_mask).float()

    if isinstance(true_mask, torch.Tensor):
        true = true_mask.detach().cpu().float()
    else:
        true = torch.from_numpy(true_mask).float()

    pred = (pred > threshold).float()
    true = (true > threshold).float()

    def _to_bchw(m):
        if m.ndim == 2:
            m = m.unsqueeze(0).unsqueeze(0)
        elif m.ndim == 3:
            m = m.unsqueeze(0)
        return m

    pred = _to_bchw(pred)
    true = _to_bchw(true)

    def get_edges(mask):
        dil = F.max_pool2d(mask, kernel_size=3, stride=1, padding=1)
        ero = 1.0 - F.max_pool2d(1.0 - mask, kernel_size=3, stride=1, padding=1)
        edge = (dil - ero > 0).float()
        return edge

    edge_pred = get_edges(pred)
    edge_true = get_edges(true)

    if edge_pred.sum() == 0 and edge_true.sum() == 0:
        return 1.0
    if edge_pred.sum() == 0 or edge_true.sum() == 0:
        return 0.0

    if tolerance > 0:
        k = 2 * tolerance + 1
        pad = tolerance
        edge_true_dil = F.max_pool2d(edge_true, kernel_size=k, stride=1, padding=pad)
        edge_pred_dil = F.max_pool2d(edge_pred, kernel_size=k, stride=1, padding=pad)
    else:
        edge_true_dil = edge_true
        edge_pred_dil = edge_pred

    precision_b = (edge_pred * edge_true_dil).sum() / (edge_pred.sum() + 1e-8)
    recall_b = (edge_true * edge_pred_dil).sum() / (edge_true.sum() + 1e-8)

    if precision_b + recall_b == 0:
        return 0.0

    bf1 = 2.0 * precision_b * recall_b / (precision_b + recall_b)
    return bf1.item()

### TUNE THESE CONSTANTS
pos_weight = 4.5
lambda_dice = 1.0

criterion_ce = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([pos_weight], device=device))

def seg_loss(logits, targets):
    return criterion_ce(logits, targets) + lambda_dice * dice_loss(logits, targets)

In [5]:
def train_model(model, optimizer, train_loader, val_loader, epochs, model_path, region_name):
    best_iou = 0.0
    patience = 10
    counter = 0

    print(f'region, epoch, train_loss, val_loss, precision, recall, f1, iou, miou, oa, bf1')

    for epoch in range(epochs):
        model.train()
        running_loss, n_samples = 0.0, 0

        for images, masks in train_loader:
            images = images.to(device, non_blocking=True)
            masks = masks.to(device, non_blocking=True).float()

            optimizer.zero_grad(set_to_none=True)

            logits = model(images)
            if logits.shape[-2:] != masks.shape[-2:]:
                logits = F.interpolate(logits, size=masks.shape[-2:], mode="bilinear", align_corners=False)

            loss = seg_loss(logits, masks)

            loss.backward()
            optimizer.step()

            running_loss += loss.item() * masks.numel()
            n_samples += masks.numel()
        
        training_loss = running_loss / max(1, n_samples)

        model.eval()
        val_loss, val_num = 0.0, 0
        
        TP, FP, FN, TN = 0,0,0,0

        bf1_sum = 0.0
        bf1_count = 0

        with torch.no_grad():
            for images, masks in val_loader:
                images = images.to(device, non_blocking=True)
                masks = masks.to(device, non_blocking=True).float()

                logits = model(images)
                if logits.shape[-2:] != masks.shape[-2:]:
                    logits = F.interpolate(logits, size=masks.shape[-2:], mode="bilinear", align_corners=False)

                loss = seg_loss(logits, masks)
                val_loss += loss.item()

                preds = torch.sigmoid(logits) > threshold

                for i in range(preds.size(0)):
                    bf1 = boundary_f1_score(preds[i, 0], masks[i, 0], tolerance=3)
                    bf1_sum += bf1
                    bf1_count += 1

                preds_flat = preds.view(-1)
                mask_flat = masks.view(-1)

                TP += ((preds_flat == 1) & (mask_flat == 1)).sum().item()
                FP += ((preds_flat == 1) & (mask_flat == 0)).sum().item()
                FN += ((preds_flat == 0) & (mask_flat == 1)).sum().item()
                TN += ((preds_flat == 0) & (mask_flat == 0)).sum().item()
                
                val_num += 1
             
        avg_bf1 = bf1_sum / max(bf1_count, 1)
        precision, recall, f1, iou, miou, oa = getMetrics(TP, TN, FP, FN)

        print(f'{region_name}, {epoch+1}, {training_loss :.3f}, {val_loss / val_num :.3f}, {precision:.4f}, {recall:.4f}, {f1:.4f}, {iou:.4f}, {miou:.4f}, {oa:.4f}, {avg_bf1:.4f}')

        if epoch == 0:
            torch.save(model.state_dict(), model_path)

        if iou > best_iou:
            best_iou = iou
            counter = 0
            
            torch.save(model.state_dict(), model_path)
        elif iou != 0.0:
            counter += 1
            if counter >= patience:
                break

def test_model(model, test_loader):
    TP, FP, FN, TN = 0,0,0,0

    model.eval()

    bf1_sum = 0.0
    bf1_count = 0

    with torch.no_grad():
        for images, masks in test_loader:
            images = images.to(device, non_blocking=True)
            masks = masks.to(device, non_blocking=True).float()

            logits = model(images)
            if logits.shape[-2:] != masks.shape[-2:]:
                logits = F.interpolate(logits, size=masks.shape[-2:], mode="bilinear", align_corners=False)

            preds = torch.sigmoid(logits) > threshold

            for i in range(preds.size(0)):
                bf1 = boundary_f1_score(preds[i, 0], masks[i, 0], tolerance=3)
                bf1_sum += bf1
                bf1_count += 1

            preds_flat = preds.view(-1)
            mask_flat = masks.view(-1)

            TP += ((preds_flat == 1) & (mask_flat == 1)).sum().item()
            FP += ((preds_flat == 1) & (mask_flat == 0)).sum().item()
            FN += ((preds_flat == 0) & (mask_flat == 1)).sum().item()
            TN += ((preds_flat == 0) & (mask_flat == 0)).sum().item()

    avg_bf1 = bf1_sum / max(bf1_count, 1)
    precision, recall, f1, iou, miou, oa = getMetrics(TP, TN, FP, FN)

    return precision, recall, f1, iou, miou, oa, avg_bf1

In [9]:
from torch.utils.data import DataLoader
from sklearn.model_selection import train_test_split
import torch.optim as optim

batch_size = 16

output = "source_region,target_region,precision,recall,f1,iou,miou,oa,bf1"

for source_region in regions_dict:
    source_images = regions_dict[source_region]
    source_masks = [f.replace("img", "mask") for f in source_images]

    train_img, val_img, train_mask, val_mask = train_test_split(
        source_images, source_masks, test_size=.2, random_state=42
    )

    for target_region in regions_dict:
        if source_region != target_region:
            epochs = 40
            model = ResNetUNet().to(device)

            train_dataset = RGBLLDataset(train_img, train_mask)
            val_dataset = LandslideDataset(val_img, val_mask)
            test_dataset = LandslideDataset(
                regions_dict[target_region],
                [f.replace("img", "mask") for f in regions_dict[target_region]],
            )            

            trainLoader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=0)
            valLoader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=0)
            testLoader = DataLoader(test_dataset, batch_size=batch_size,shuffle=False,num_workers=0)

            optimizer = optim.Adam(model.parameters(), lr=1e-4, weight_decay=1e-5)

            model_path = f"../../models/new_wavelet/rgbll{source_region}_{target_region}.pth"

            train_model(model, optimizer, trainLoader, valLoader, epochs, model_path, source_region)

            model.load_state_dict(torch.load(model_path, map_location=device))

            precision, recall, f1, iou, miou, oa, bf1 = test_model(model, testLoader)
            result = f'{source_region}, {target_region}, {precision:.4f}, {recall:.4f}, {f1:.4f}, {iou:.4f}, {miou:.4f}, {oa:.4f}, {bf1:.4f}'
            print(result)
            output += f'\n{result}'

with open(f"../../results/new_wavelet/rgbll.csv", "w") as f:
    f.write(output)


region, epoch, train_loss, val_loss, precision, recall, f1, iou, miou, oa, bf1


RuntimeError: Given groups=1, weight of size [64, 6, 7, 7], expected input[16, 3, 512, 512] to have 6 channels, but got 3 channels instead

In [7]:
from torch.utils.data import DataLoader
from sklearn.model_selection import train_test_split
import torch.optim as optim

batch_size = 16

output = "source_region,target_region,precision,recall,f1,iou,miou,oa,bf1"

for source_region in regions_dict:
    source_images = regions_dict[source_region]
    source_masks = [f.replace("img", "mask") for f in source_images]

    train_img, val_img, train_mask, val_mask = train_test_split(
        source_images, source_masks, test_size=.2, random_state=42
    )

    for target_region in regions_dict:
        if source_region != target_region:
            epochs = 40
            model = ResNetUNet().to(device)

            train_dataset = HFOnlyDataset(train_img, train_mask)
            val_dataset = LandslideDataset(val_img, val_mask)
            test_dataset = LandslideDataset(
                regions_dict[target_region],
                [f.replace("img", "mask") for f in regions_dict[target_region]],
            )            

            trainLoader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=0)
            valLoader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=0)
            testLoader = DataLoader(test_dataset, batch_size=batch_size,shuffle=False,num_workers=0)

            optimizer = optim.Adam(model.parameters(), lr=1e-4, weight_decay=1e-5)

            model_path = f"../../models/new_wavelet/hf/{source_region}_{target_region}.pth"

            train_model(model, optimizer, trainLoader, valLoader, epochs, model_path, source_region)

            model.load_state_dict(torch.load(model_path, map_location=device))

            precision, recall, f1, iou, miou, oa, bf1 = test_model(model, testLoader)
            result = f'{source_region}, {target_region}, {precision:.4f}, {recall:.4f}, {f1:.4f}, {iou:.4f}, {miou:.4f}, {oa:.4f}, {bf1:.4f}'
            print(result)
            output += f'\n{result}'

with open(f"../../results/new_wavelet/hf.csv", "w") as f:
    f.write(output)


region, epoch, train_loss, val_loss, precision, recall, f1, iou, miou, oa, bf1
Hokkaido Iburi-Tobu, 1, 1.774, 1.730, 0.0000, 0.0000, 0.0000, 0.0000, 0.4686, 0.9372, 0.0000
Hokkaido Iburi-Tobu, 2, 1.761, 1.716, 0.0000, 0.0000, 0.0000, 0.0000, 0.4686, 0.9372, 0.0000
Hokkaido Iburi-Tobu, 3, 1.744, 1.690, 0.0000, 0.0000, 0.0000, 0.0000, 0.4686, 0.9372, 0.0000
Hokkaido Iburi-Tobu, 4, 1.725, 1.648, 0.0000, 0.0000, 0.0000, 0.0000, 0.4686, 0.9372, 0.0000
Hokkaido Iburi-Tobu, 5, 1.706, 1.597, 0.0000, 0.0000, 0.0000, 0.0000, 0.4686, 0.9372, 0.0000
Hokkaido Iburi-Tobu, 6, 1.700, 1.523, 0.0000, 0.0000, 0.0000, 0.0000, 0.4686, 0.9372, 0.0000
Hokkaido Iburi-Tobu, 7, 1.667, 1.996, 0.0000, 0.0000, 0.0000, 0.0000, 0.4686, 0.9372, 0.0000
Hokkaido Iburi-Tobu, 8, 1.656, 1.948, 0.0000, 0.0000, 0.0000, 0.0000, 0.4686, 0.9372, 0.0000
Hokkaido Iburi-Tobu, 9, 1.583, 1.679, 0.0000, 0.0000, 0.0000, 0.0000, 0.4686, 0.9372, 0.0000
Hokkaido Iburi-Tobu, 10, 1.521, 2.361, 0.0000, 0.0000, 0.0000, 0.0000, 0.4686, 0.937